# Module 2: Ensemble Methods — Hands-On Exercise
## Machine learning for energy applications — Module 2, Exercise 1

In this exercise, we will explore **ensemble methods** — techniques that combine multiple models
to produce stronger predictions than any single model alone. We will work with the classic Iris
dataset and build progressively more powerful classifiers.

---
## Overview

**Topics covered:**

1. Data preparation (Iris dataset, label mapping, stratified train/test split)
2. Single Decision Tree baseline
3. Random Forest classifier
4. Gradient Boosting classifier
5. Model comparison with cross-validation on the training rows, then one final test evaluation
6. Feature importance: impurity (MDI) versus permutation importance
7. Building an sklearn Pipeline (StandardScaler + model), saving and reloading it
8. Hyperparameter tuning with GridSearchCV
9. (Optional) XGBoost, or scikit-learn's HistGradientBoostingClassifier

**Key idea:** A single decision tree is prone to overfitting, but by aggregating many trees —
either through *bagging* (Random Forest) or *boosting* (Gradient Boosting) — we can often
improve generalization. Whether they do on a particular dataset is an empirical question,
answered with validation evidence and the spread across folds.

**Evaluation rules in this notebook:**
- Models are fitted, compared and tuned on the 120 **training** rows only, with 5-fold
  cross-validation (the same folds for every model).
- The 30 **test** rows are opened only in clearly labelled cells, after the models are fixed:
  the final evaluation of Lab 1 (end of Part 5), the permutation importance of the already
  evaluated forest (Part 6) and the final evaluation of Lab 2 (end of Part 8). No model,
  setting or feature is chosen from them.

In [ ]:
import tempfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['figure.dpi'] = 100

---
## Part 1: Data Preparation

Load the Iris dataset from `sklearn.datasets`, inspect its structure, print the label mapping,
and split into training (80%) and test (20%) sets.

The split is stratified so that each class keeps its share in both parts. From here on the
**test rows are sealed**: we do not score them until the final evaluation at the end of Part 5.

In [ ]:
# Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"Dataset shape: {X.shape}")
print(f"Number of features: {X.shape[1]}")
print(f"Feature names: {feature_names}")
label_mapping = {i: str(name) for i, name in enumerate(target_names)}
print(f"Label mapping (code -> class): {label_mapping}")
print(f"\nClass distribution:")
for i, name in enumerate(target_names):
    print(f"  {i} = {name}: {np.sum(y == i)} samples")

# Train/test split (80/20), stratified. The test rows stay sealed until the final evaluation.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples (sealed until the final evaluation)")

---
## Part 2: Single Decision Tree

Train a single `DecisionTreeClassifier` on the training rows as a baseline. This will help us
see how much improvement the ensemble methods provide.

We do not score the test rows here. Training accuracy only shows how well a model fits the rows
it has seen; the models are compared with cross-validation on the training rows in Part 5.

In [ ]:
# Train a single Decision Tree on the training rows
dt_clf = DecisionTreeClassifier(random_state=42)
dt_clf.fit(X_train, y_train)

# Training accuracy is optimistic: an unpruned tree can memorize its training rows.
# It is not evidence about new data (see the cross-validation in Part 5).
acc_dt_train = accuracy_score(y_train, dt_clf.predict(X_train))
print(f"Decision Tree training accuracy: {acc_dt_train:.4f}")
print(f"Tree depth: {dt_clf.get_depth()}, leaves: {dt_clf.get_n_leaves()}")

---
## Part 3: Random Forest

Train a `RandomForestClassifier` with 100 estimators on the training rows. Random Forest is a
*bagging* method: it trains many decision trees on bootstrap samples of the rows, with a random
subset of features considered at each split, then averages the trees' class probabilities (a soft vote).

In [ ]:
# Train a Random Forest on the training rows
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)

# Training accuracy (optimistic, like the tree's)
acc_rf_train = accuracy_score(y_train, rf_clf.predict(X_train))
print(f"Random Forest training accuracy: {acc_rf_train:.4f}")

---
## Part 4: Gradient Boosting

Train a `GradientBoostingClassifier` with 100 estimators on the training rows. Gradient Boosting
is a *boosting* method: it trains trees sequentially, where each new tree focuses on correcting
the errors made by the previous ensemble.

In [ ]:
# Train a Gradient Boosting classifier on the training rows
gb_clf = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_clf.fit(X_train, y_train)

# Training accuracy (optimistic)
acc_gb_train = accuracy_score(y_train, gb_clf.predict(X_train))
print(f"Gradient Boosting training accuracy: {acc_gb_train:.4f}")

---
## Part 5: Model Comparison

Compare all three models using **5-fold cross-validation on the training rows only**
(`X_train`, `y_train`). The test rows must stay out of the folds: a score computed with the
test rows inside the folds cannot also be reported as independent test evidence.

With a classifier, `cv=5` means `StratifiedKFold(n_splits=5)` without shuffling, so every model
is scored on the same five folds and we can compare them fold by fold. We will visualize the
results with a bar chart.

In [ ]:
# Cross-validation for all three models on the TRAINING rows only (same 5 folds for every model)
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name:20s} — Mean CV Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Fold by fold: are the differences between models larger than the differences between folds?
print("\nAccuracy per fold (24 validation rows per fold, so one row = 0.042):")
print(pd.DataFrame(cv_results, index=[f"fold {k}" for k in range(1, 6)]).round(3))

# Bar chart of mean cross-validation accuracies
model_names = list(cv_results.keys())
mean_scores = [cv_results[name].mean() for name in model_names]
std_scores = [cv_results[name].std() for name in model_names]

plt.figure(figsize=(9, 5))
bars = plt.bar(model_names, mean_scores, yerr=std_scores, capsize=5,
               color=['#2196F3', '#4CAF50', '#FF9800'], edgecolor='black', linewidth=1.2)

# Add value labels on bars
for bar, score in zip(bars, mean_scores):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f'{score:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.ylim(0.85, 1.02)
plt.ylabel('Mean CV Accuracy (training rows)')
plt.title('Model Comparison — 5-Fold Cross-Validation on the Training Rows')
plt.tight_layout()
plt.show()

### 5.1 Extension (optional): out-of-bag accuracy and a validation curve

For fast groups; both checks use the training rows only.
- **Out-of-bag (OOB) accuracy** scores each training row with the trees whose bootstrap sample
  left it out. Like ordinary cross-validation, it assumes the rows are unrelated.
- **Validation curve for boosting:** split the training rows into training and validation rows,
  then plot validation accuracy against the number of boosting stages with `staged_predict`.

In [ ]:
# Out-of-bag accuracy of the same forest (same random_state, so the same trees as rf_clf)
rf_oob = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)
rf_oob.fit(X_train, y_train)
print(f"Random Forest OOB accuracy (training rows): {rf_oob.oob_score_:.4f}")
print(f"Random Forest 5-fold CV mean (Part 5):      {cv_results['Random Forest'].mean():.4f}")

# Validation curve: split the TRAINING rows into training and validation rows (test rows untouched)
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train
)
gb_curve = GradientBoostingClassifier(n_estimators=100, random_state=42).fit(X_fit, y_fit)
val_acc = [accuracy_score(y_val, y_hat) for y_hat in gb_curve.staged_predict(X_val)]

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(val_acc) + 1), val_acc, color='#FF9800')
plt.ylim(0.8, 1.01)   # one validation row is 1/30 = 0.033: do not over-read small steps
plt.xlabel('Number of boosting stages')
plt.ylabel(f'Validation accuracy ({len(y_val)} rows)')
plt.title('Gradient Boosting — validation accuracy by stage')
plt.tight_layout()
plt.show()
print(f"Best validation accuracy {max(val_acc):.3f}, first reached at stage {int(np.argmax(val_acc)) + 1}; "
      f"after all {len(val_acc)} stages: {val_acc[-1]:.3f}")

### 5.2 Final evaluation of Lab 1: open the test rows once

First write down your conclusion from the cross-validation (which model, or "no clear
difference"). Then score the three models from Parts 2–4, each fitted on all 120 training
rows, on the 30 sealed test rows — once. One test row is worth 1/30 = 3.3 percentage points,
so compare any difference with the fold spread above. Nothing is re-tuned or re-chosen after
this cell.

In [ ]:
# FINAL EVALUATION (Lab 1): the three models fitted in Parts 2-4, scored once on the test rows
fitted_models = {'Decision Tree': dt_clf, 'Random Forest': rf_clf, 'Gradient Boosting': gb_clf}

rows = []
for name, model in fitted_models.items():
    y_pred = model.predict(X_test)
    rows.append({
        'Model': name,
        'CV mean (training rows)': cv_results[name].mean(),
        'CV std': cv_results[name].std(),
        'Test accuracy': accuracy_score(y_test, y_pred),
        'Test rows correct': f"{(y_pred == y_test).sum()}/{len(y_test)}",
    })
    print(f"--- {name}: classification report on the test rows")
    print(classification_report(y_test, y_pred, target_names=target_names))

final_table = pd.DataFrame(rows).set_index('Model')
final_table.round(3)

---
## Part 6: Feature Importance

Tree-based ensembles provide **impurity importance** (mean decrease in impurity, MDI) in
`feature_importances_`. It is computed from the training splits and rewards any feature the
trees happened to split on. **Permutation importance** instead measures how much a held-out
score drops when one feature is shuffled: the reliance of a fitted model on that feature.

The forest from Part 3 is already fixed and was evaluated at the end of Part 5, so here we may
use the test rows to *interpret* it. With 30 rows the values are noisy, and no feature may be
dropped or selected on the basis of this table. Neither measure is a causal effect.

In [ ]:
# Impurity importance (MDI) of the Random Forest, computed from the training data
importances = rf_clf.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(8, 5))
plt.barh(range(len(indices)), importances[indices], color='#4CAF50', edgecolor='black')
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel('Impurity importance (MDI, training data)')
plt.title('Random Forest — Feature Importance (Iris Dataset)')
plt.tight_layout()
plt.show()

# Print importance values
print("Feature Importances (descending):")
for i in reversed(indices):
    print(f"  {feature_names[i]:25s}: {importances[i]:.4f}")

In [ ]:
# Permutation importance of the fixed forest on the test rows (interpretation only)
perm = permutation_importance(rf_clf, X_test, y_test, scoring='accuracy',
                              n_repeats=20, random_state=0)

importance_table = pd.DataFrame({
    'MDI (training)': rf_clf.feature_importances_,
    'Permutation mean (test, 20 repeats)': perm.importances_mean,
    'Permutation std': perm.importances_std,
}, index=feature_names).sort_values('MDI (training)', ascending=False)
importance_table.round(3)

---
## Part 7: sklearn Pipeline

An sklearn `Pipeline` chains preprocessing and modelling steps into a single object.
This ensures that all transformations (e.g. scaling) are applied consistently during
both training and prediction, and prevents data leakage during cross-validation: the scaler
is refitted inside every training fold.

We will build a pipeline that:
1. Scales the features with `StandardScaler`
2. Classifies with `RandomForestClassifier`

For trees, scaling changes nothing (splits depend only on the order of the values); we keep the
scaler to practise packaging preprocessing with the model. We cross-validate the pipeline on the
training rows, then save it with `joblib`, reload it, and check that the reloaded pipeline
reproduces the predictions on five reference rows.

In [ ]:
# Create a pipeline: StandardScaler -> RandomForestClassifier
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

print("Pipeline steps:")
for name, step in pipe.steps:
    print(f"  {name}: {step.__class__.__name__}")

In [ ]:
# Fit the pipeline on the training rows
pipe.fit(X_train, y_train)

# 5-fold cross-validation of the pipeline on the training rows (same folds as Part 5)
cv_pipe = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
print(f"Pipeline 5-Fold CV Accuracy (training rows): {cv_pipe.mean():.4f} (+/- {cv_pipe.std():.4f})")
print(f"Random Forest without scaler (Part 5):       "
      f"{cv_results['Random Forest'].mean():.4f} (+/- {cv_results['Random Forest'].std():.4f})")

In [ ]:
# Save the fitted pipeline (scaler and forest together) with joblib.
# A scratch folder keeps the course folder clean; in a project, save to a versioned folder.
model_path = Path(tempfile.mkdtemp()) / 'iris_rf_pipeline.joblib'
joblib.dump(pipe, model_path)

# A fixed reference batch (features only) and the predictions of the pipeline before saving
reference_rows = X_test[:5]
reference_pred = pipe.predict(reference_rows)
reference_proba = pipe.predict_proba(reference_rows)
print(f"Saved to: {model_path}")
print(f"Reference predictions: {reference_pred}")

In [ ]:
# Reload in a new cell, as a deployment script would (load only files from trusted sources)
reloaded = joblib.load(model_path)

# Do NOT call fit on the reloaded model: it must reproduce the saved model's predictions
assert np.array_equal(reloaded.predict(reference_rows), reference_pred)
assert np.allclose(reloaded.predict_proba(reference_rows), reference_proba)
print("Reload check passed on 5 reference rows:",
      [str(target_names[k]) for k in reloaded.predict(reference_rows)])

---
## Part 8: Hyperparameter Tuning with GridSearchCV

Use `GridSearchCV` to search over a grid of hyperparameters for the Random Forest
inside the pipeline. When referencing pipeline parameters, use the format
`stepname__parametername` (double underscore).

We will tune:
- `clf__n_estimators`: \[50, 100, 200\]
- `clf__max_depth`: \[None, 5, 10\]
- `clf__min_samples_split`: \[2, 5\]

The search selects on 5-fold cross-validation of the **training rows**. In `cv_results_`,
"test" (as in `mean_test_score`) means the held-out fold of each split, i.e. validation data,
not our sealed test rows. Inspect every row: how many settings share the best score?

In [ ]:
# Define the parameter grid for the pipeline
param_grid = {
    'clf__n_estimators': [50, 100, 200],
    'clf__max_depth': [None, 5, 10],
    'clf__min_samples_split': [2, 5]
}

print(f"Total parameter combinations: {3 * 3 * 2} = "
      f"{3 * 3 * 2 * 5} fits (with 5-fold CV)")

In [ ]:
# Run GridSearchCV on the training rows
grid_search = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

# Print results
print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Score: {grid_search.best_score_:.4f}")

In [ ]:
# Inspect every row of cv_results_, not only best_params_
results = pd.DataFrame(grid_search.cv_results_)
shown = ['param_clf__n_estimators', 'param_clf__max_depth', 'param_clf__min_samples_split',
         'mean_test_score', 'std_test_score', 'rank_test_score']
print(results[shown].sort_values('rank_test_score', kind='stable').to_string(index=False))

n_best = int((results['rank_test_score'] == 1).sum())
print(f"\n{n_best} of {len(results)} settings share the best mean CV accuracy "
      f"({grid_search.best_score_:.4f}); best_params_ is the first of them in the grid.")

### 8.1 Final evaluation of Lab 2: the locked choice on the test rows

`GridSearchCV` has refitted the selected setting on all training rows (`best_estimator_`).
That choice is now locked, and it is scored once on the sealed test rows. Nothing is tuned after
this cell; if you wanted to try another grid, the test score would no longer be an independent
estimate.

In [ ]:
# FINAL EVALUATION (Lab 2): the locked choice, scored once on the test rows
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
acc_best = accuracy_score(y_test, y_pred_best)
print(f"Best Model Test Accuracy: {acc_best:.4f} ({(y_pred_best == y_test).sum()}/{len(y_test)} rows)")
print(f"\nClassification Report (Best Model):")
print(classification_report(y_test, y_pred_best, target_names=target_names))

---
## Part 9: Advanced — XGBoost (Optional)

XGBoost is a popular, high-performance gradient boosting library. It is an optional package
(listed in `requirements-labs-optional.txt` at the course root). If it is installed, we compare
it with the scikit-learn models; otherwise the cell uses scikit-learn's built-in
`HistGradientBoostingClassifier`.

XGBoost 2 and later ignore `use_label_encoder` (with a warning), so the cell leaves it out:
the labels must already be integers 0, 1, 2 (as here), and `eval_metric` is set in the constructor.
The comparison uses the same 5 folds of the training rows as Part 5; the sealed test rows are
not reopened for this extra model.

In [ ]:
# Optional: XGBoost if installed, otherwise scikit-learn's HistGradientBoostingClassifier
try:
    from xgboost import XGBClassifier
    booster = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3,
                            eval_metric='mlogloss', random_state=42)
    booster_name = 'XGBoost'
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier
    print("xgboost is not installed (optional: pip install xgboost).")
    print("Using scikit-learn's HistGradientBoostingClassifier instead.\n")
    booster = HistGradientBoostingClassifier(learning_rate=0.1, max_depth=3, random_state=42)
    booster_name = 'HistGradientBoosting'

# Same 5 folds of the training rows as Part 5; the test rows stay sealed
cv_boost = cross_val_score(booster, X_train, y_train, cv=5, scoring='accuracy')
print(f"{booster_name} 5-Fold CV Accuracy (training rows): {cv_boost.mean():.4f} (+/- {cv_boost.std():.4f})")
for name, scores in cv_results.items():
    print(f"{name} (Part 5): {scores.mean():.4f} (+/- {scores.std():.4f})")

---
## Summary

In this exercise we:

1. **Loaded and explored** the Iris dataset (150 samples, 4 features, 3 classes), printed the
   label mapping, and made a stratified 80/20 split with the test rows sealed.
2. Trained a **single Decision Tree** as a baseline classifier.
3. Trained a **Random Forest** (bagging) and **Gradient Boosting** (boosting) classifier.
4. **Compared** all three models with 5-fold cross-validation on the training rows (same folds),
   then opened the test rows once for the final evaluation of Lab 1.
5. Compared **impurity (MDI) and permutation importance** for the fixed Random Forest.
6. Built an sklearn **Pipeline** combining `StandardScaler` and `RandomForestClassifier`, saved it,
   reloaded it, and checked that it reproduces its predictions.
7. Used **GridSearchCV** on the training rows, inspected every row of `cv_results_`, and scored
   the locked choice once on the test rows.
8. (Optional) Compared **XGBoost** (or `HistGradientBoostingClassifier`) on the same folds.

**Key takeaways:**
- Bagging averages away much of the variance of deep trees, and boosting builds a flexible
  model from many small corrections; but on a small, nearly separable dataset such as Iris the differences between models are within the fold-to-fold
  noise. Check the spread before ranking models.
- Choose models and settings with cross-validation on the training rows (or a validation
  split); open the test rows once, for the locked choice.
- MDI is a quick training-data diagnostic; permutation importance on held-out rows measures how
  much a fitted model relies on a feature. Neither is a causal effect.
- sklearn Pipelines keep preprocessing inside every cross-validation fold and travel with the
  model when it is saved; check that a reloaded model reproduces its predictions.
- GridSearchCV selects on cross-validation scores; look at `cv_results_` for ties before trusting
  `best_params_`.